# Meta gene modules (MGMs) from ten ARCHS4 CLAMPfull models

Report of the CRISPR-Cas MGM analysis: technical noise screen of the LVs of the canonical model and nine seed replicates, clustering into MGMs (cosine >= 0.5, kept when present in >= 6 of 10 models), ORA, selection with the CRISPR lipid screen, PhenoPLIER traits and canonical pathways, sample projection, and the LINCS L1000 drugs of the four Figure 5 MGMs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    I = snakemake.input
    FLAGS, SUMMARY, FUNNEL, PATHWAYS = Path(I.flags), Path(I.summary), Path(I.funnel), Path(I.pathways)
    MODULES, TRAITS, COMPOSITION, RANKING = Path(I.modules), Path(I.traits), Path(I.composition), Path(I.ranking)
    ORA = [Path(d) for d in I.ora]
except NameError:
    A4 = Path('../../../../output/03_model_biology/02_archs4')
    M = A4 / '04_crispercas/00_mgm'
    FLAGS, SUMMARY, FUNNEL, PATHWAYS = A4 / '06_technical_noise/seeds/all_flags.csv', M / 'cluster_summary.csv', M / 'selection/funnel.csv', M / 'pathways/canonical_bp_top.csv'
    MODULES, TRAITS, COMPOSITION, RANKING = M / 'fig5/modules.csv', M / 'fig5/module_trait_groups.csv', M / 'projection/composition.csv', M / 'l1000/l1000_fig5_ranking.csv'
    ORA = [M / 'ora' / d for d in ['bp', 'canonical', 'cellmarker', 'azimuth', 'archs4_cell_lines']]

def mgm(s):
    return s.str.replace('metaLV', 'MGM')

ORDER = ['MGM91', 'MGM588', 'MGM175', 'MGM335']
PICK = {'MGM91': 'Clomifene', 'MGM588': 'Clomifene', 'MGM175': 'Cortisone acetate', 'MGM335': 'Calcitriol'}

In [ ]:
flags = pd.read_csv(FLAGS)
noise = flags.groupby('model', sort=False).agg(lvs=('LV', 'size'), gene_level=('gene_flag', 'sum'),
                                               sample_level=('sample_flag', 'sum'), removed=('any_flag', 'sum'))
noise['kept'] = noise.lvs - noise.removed
noise.loc['total'] = noise.sum()
display(noise)

In [ ]:
clusters = pd.read_csv(SUMMARY)
clusters['MGM'] = mgm(clusters.metaLV)
dist = pd.read_csv(SUMMARY.parent / 'cluster_model_count_distribution.csv', index_col=0)['count']
print(f"{noise.loc['total', 'kept']} LVs -> {dist.sum()} clusters -> {len(clusters)} MGMs present in >= 6/10 models")
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.bar(dist.index, dist.values, color=['#BBBBBB' if n < 6 else '#2A7AB0' for n in dist.index])
ax.set(xlabel='models in cluster', ylabel='clusters', yscale='log', xticks=dist.index)
plt.tight_layout()
plt.show()
display(clusters[['n_lvs', 'n_models', 'min_cos', 'mean_cos']].describe().T.round(2))

In [ ]:
ora = pd.concat([pd.read_csv(d / 'summary.csv') for d in ORA], ignore_index=True)
ora['MGMs_enriched'] = [pd.read_csv(d / 'enrichment.csv.gz', usecols=['LV', 'p.adjust']).query('`p.adjust` < 0.05').LV.nunique() for d in ORA]
display(ora.set_index('database_label')[['tested_lvs', 'eligible_pathways', 'recovered_pathways', 'recovered_percent', 'MGMs_enriched']].round(1))

In [ ]:
funnel = pd.read_csv(FUNNEL)
STEPS = {'pass_gsea': 'CRISPR lipid screen genes enriched (GSEA FDR < 0.05)',
         'pass_trait': 'PhenoPLIER trait association (FDR < 0.05)',
         'pass_canonical': 'canonical pathway enriched (ORA FDR < 0.05)',
         'pass_screen_gene_top': 'screen gene in the top 185 loadings',
         'pass_missed_by_twas': 'screen gene not TWAS-associated with the trait (P >= 0.05)',
         'pass_lipid_pathway': 'lipid canonical pathway'}
steps = pd.Series({'clustered in >= 6/10 models': len(funnel)} | {v: int(funnel[k].sum()) for k, v in STEPS.items()}, name='MGMs')
display(steps.to_frame())
display(funnel[funnel.pass_gsea].assign(MGM=lambda d: mgm(d.metaLV)).sort_values('gsea_padj')
        [['MGM', 'n_models', 'gsea_padj', 'crispr_leading_edge'] + list(STEPS)[1:]].style.format({'gsea_padj': '{:.1e}'}).hide(axis='index'))

In [ ]:
modules = pd.read_csv(MODULES).assign(MGM=lambda d: mgm(d.metaLV)).set_index('MGM')
traits = pd.read_csv(TRAITS).assign(MGM=lambda d: mgm(d.metaLV))
traits = traits.groupby('MGM').apply(lambda d: '; '.join(f"{g} ({q:.1e})" for g, q in zip(d.trait_group, d.fdr)), include_groups=False).rename('traits (best FDR)')
display(modules[['models', 'cells', 'pathways']].join(traits).join(modules[['drug', 'drug_rank', 'drug_fdr']]).loc[ORDER])

In [ ]:
top = pd.read_csv(PATHWAYS).assign(MGM=lambda d: mgm(d.metaLV))
top = top.sort_values('padj').groupby('MGM').head(5)
for m in ORDER:
    print(m)
    display(top[top.MGM == m][['db', 'pathway', 'padj']].style.format({'padj': '{:.1e}'}).hide(axis='index'))

In [ ]:
comp = pd.read_csv(COMPOSITION).assign(MGM=lambda d: mgm(d.metaLV))
comp = comp[comp.set == 'top 1%'].sort_values('pct', ascending=False).groupby('MGM').head(3)
display(comp.groupby('MGM').apply(lambda d: '; '.join(f"{c} {p:.0f}%" for c, p in zip(d.category, d.pct)), include_groups=False)
        .rename('top 1% samples by B (top 3 categories)').loc[ORDER].to_frame())

In [ ]:
r = pd.read_csv(RANKING)
r['MGM'] = mgm(r.metaLV)
assert set(r.MGM) == set(ORDER) and r.groupby('MGM').size().eq(1170).all()

In [ ]:
def has(s, *keys):
    return s.fillna('').str.contains('|'.join(keys), case=False, regex=True)

sterol = ['statin$', 'conazole$', 'Clomifene', 'Tamoxifen', 'Amiodarone', 'Haloperidol', 'Trazodone', 'Aripiprazole', 'Trimipramine']
LINK = {
    'sterol': lambda d: has(d.drug, *sterol) & ~has(d.drug, 'Bestatin', 'Cilastatin', 'Wiskostatin', 'blebbistatin', 'Alrestatin')
                        | has(d.atc_codes, 'N05A', 'N06A'),
    'MGM175': lambda d: has(d.categories, 'Glucocorticoid', 'Corticosteroid') | has(d.drug, 'cholic', 'Ursodeoxycholic', 'fibrate$', 'Gemfibrozil', 'glitazone', 'PREGN-4-ENE')
                        | has(d.atc_codes, 'G03C'),
    'MGM335': lambda d: has(d.drug, 'Calcitriol', 'DN-101', 'Vitamin A', 'Tretinoin') | has(d.categories, 'Glucocorticoid', 'Anti-Inflammatory Agents, Non-Steroidal')
                        | has(d.atc_codes, 'L04'),
}
r['linked'] = False
for m in ORDER:
    i = r.MGM == m
    r.loc[i, 'linked'] = LINK['sterol' if m in ('MGM91', 'MGM588') else m](r[i]).values

In [ ]:
picks = pd.concat([r[(r.MGM == m) & (r.drug == d)] for m, d in PICK.items()])
summary = r.groupby('MGM').agg(n_genes=('n_genes', 'first'), n_fdr_0_05=('fdr', lambda q: (q < 0.05).sum()),
                               linked_top10=('linked', lambda s: s.iloc[:10].sum()),
                               linked_top1pct=('linked', lambda s: s.iloc[:12].sum()))
display(picks.set_index('MGM')[['drug', 'rank', 'score', 'perm_z', 'perm_p', 'fdr']].join(summary).loc[ORDER])

In [ ]:
cols = ['rank', 'drug', 'drugbank_id', 'score', 'mean_z', 'perm_z', 'fdr', 'linked']
for m in ORDER:
    print(m)
    display(r[r.MGM == m].head(10)[cols].style.format({'score': '{:.2f}', 'mean_z': '{:.2f}', 'perm_z': '{:.1f}', 'fdr': '{:.1e}'})
            .apply(lambda row: ['font-weight: bold' if row.drug == PICK[m] else '' for _ in row], axis=1).hide(axis='index'))